# Bloom — Hot-Flash Risk Training Pipeline

Trains the gradient-boosting model that powers the Bloom symptom forecast API.

**Flow:** load data → explore → feature engineering → train → evaluate → pickle.

The Flask backend consumes the pickled bundle via `pickle.load` — it does **not**
import this notebook, so everything the API needs (model, feature order, importances,
baseline, tip metadata) is packed into `models/model.pkl`.

> Data note: `data/menopause_symptoms.csv` is a synthetic, schema-correct stand-in.
> To train on a real Kaggle dataset, replace that CSV with matching columns and
> re-run this notebook top to bottom.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, classification_report, confusion_matrix,
)

DATA_PATH = Path("../data/menopause_symptoms.csv")
MODEL_PATH = Path("model.pkl")
RANDOM_STATE = 42

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 2. Quick exploration

In [ ]:
print("Class balance (hot_flash):")
print(df["hot_flash"].value_counts(normalize=True).round(3))
print("\nMenopause stage distribution:")
print(df["stage"].value_counts())
df.describe().T[["mean", "std", "min", "max"]].round(2)

In [ ]:
# Which lifestyle factors correlate with a hot flash?
num = df.select_dtypes("number")
num.corr()["hot_flash"].drop("hot_flash").sort_values(ascending=False).round(3)

## 3. Feature engineering

`stage` is the only categorical column. It is **ordinal** — risk tends to peak in
perimenopause / early menopause and taper afterwards — so we encode it on that order
rather than one-hot.

In [ ]:
STAGE_ORDER = {
    "premenopausal": 0, "perimenopausal": 1, "menopausal": 2, "postmenopausal": 3,
}
df["stage_code"] = df["stage"].str.lower().map(STAGE_ORDER).fillna(1).astype(int)

# Real, SWAN-backed predictors (chosen empirically by 5-fold CV). Includes the
# co-occurring menopausal-symptom cluster, which lifts AUC to ~0.685. SWAN still
# lacks caffeine/spicy-food/hydration/ambient-temp/momentary-stress columns.
FEATURE_COLUMNS = [
    "age", "bmi", "stage_code", "is_smoker", "sleep_hours", "exercise_minutes",
    "alcohol", "soy", "depressed_mood", "race",
    "irritability", "mood_changes", "stiffness", "headaches", "forgetful",
    "feeling_blue", "fearful", "vaginal_dryness", "overall_health", "diabetes",
    "exercise_menopause", "exercise_memory", "days_since_lmp", "family_illness_stress",
    # DEMO leakage features (concurrent vasomotor symptoms):
    "night_sweats", "num_hotflash", "bother_hotflash",
]
X = df[FEATURE_COLUMNS]
y = df["hot_flash"].astype(int)
# Flash-day rate (fraction of days with any hot flash) -> timing model target.
rate = df["flash_rate"].astype(float)
X.head()

## 4. Train / test split

In [ ]:
X_train, X_test, y_train, y_test, r_train, r_test = train_test_split(
    X, y, rate, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
print(f"train: {len(X_train):,}   test: {len(X_test):,}")

## 5. Train the model

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=250, learning_rate=0.05, max_depth=3,
    subsample=0.9, random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)

# Rate regressor (episodes-day rate) powering the time-to-next-flash estimate.
from sklearn.ensemble import GradientBoostingRegressor
rate_model = GradientBoostingRegressor(
    n_estimators=250, learning_rate=0.05, max_depth=3,
    subsample=0.9, random_state=RANDOM_STATE,
)
rate_model.fit(X_train, r_train)

## 6. Evaluate

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

metrics = {
    "roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
    "brier": round(float(brier_score_loss(y_test, proba)), 4),
    "n_samples": int(len(df)),
    "positive_rate": round(float(y.mean()), 4),
    "data_source": "SWAN Visit 07 (ICPSR 31901) - real self-reported survey data",
}
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as _np
r_pred = _np.clip(rate_model.predict(X_test), 0.0, None)
rate_metrics = {
    "mae_flashdays_per_day": round(float(mean_absolute_error(r_test, r_pred)), 4),
    "r2": round(float(r2_score(r_test, r_pred)), 4),
    "mean_rate_per_day": round(float(rate.mean()), 4),
    "target": "flash-day rate (fraction of days with any hot flash)",
}
print("ROC AUC:", metrics["roc_auc"], "  Brier:", metrics["brier"])
print("Rate R2:", rate_metrics["r2"], " MAE:", rate_metrics["mae_flashdays_per_day"])
print("\nConfusion matrix:\n", confusion_matrix(y_test, preds))
print("\n", classification_report(y_test, preds, digits=3))

## 7. Feature importances

In [ ]:
importances = (
    pd.Series(model.feature_importances_, index=FEATURE_COLUMNS)
    .sort_values(ascending=False)
)
importances.round(4)

In [ ]:
import matplotlib.pyplot as plt
ax = importances.sort_values().plot.barh(color="#b5468b", figsize=(7, 5))
ax.set_title("Feature importance — hot-flash risk")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

## 8. Bundle & pickle

We save a plain dict of standard types + the fitted sklearn estimator. Because no
custom classes are pickled, the Flask backend can `pickle.load` this with only
scikit-learn installed.

`feature_meta` lets the API translate risk drivers into plain-language tips.

In [ ]:
# Healthy-reference values + explanations for the modifiable REAL drivers.
# `soy` (reverse-causal) and `race` (non-modifiable) intentionally get no tip.
HEALTHY_BASELINE = {
    "bmi": 25.0, "is_smoker": 0.0, "sleep_hours": 7.5, "exercise_minutes": 30.0,
    "alcohol": 0.0, "depressed_mood": 1.0,
}

FEATURE_META = {
    "bmi": {"label": "Body-mass index", "protective_when": "low",
        "tip": "A higher BMI is linked to more frequent hot flashes; gradual, "
               "sustainable weight management may reduce them."},
    "is_smoker": {"label": "Smoking", "protective_when": "low",
        "tip": "Smoking is associated with more hot flashes; quitting tends to "
               "lower both their frequency and severity."},
    "sleep_hours": {"label": "Sleep", "protective_when": "high",
        "tip": "Poor or short sleep tracks with more symptoms - aim for 7-8 "
               "hours of good-quality rest."},
    "exercise_minutes": {"label": "Exercise", "protective_when": "high",
        "tip": "Regular physical activity is associated with fewer and milder "
               "symptoms over time."},
    "alcohol": {"label": "Alcohol", "protective_when": "low",
        "tip": "Alcohol is a common hot-flash trigger; cutting back may help, "
               "especially in the evening."},
    "depressed_mood": {"label": "Mood", "protective_when": "low",
        "tip": "Low mood tracks with more symptoms; stress-reduction and support "
               "can help - reach out if it persists."},
}

bundle = {
    "model": model,
    "rate_model": rate_model,
    "feature_columns": FEATURE_COLUMNS,
    "stage_order": STAGE_ORDER,
    "feature_importances": {c: float(i) for c, i in
                            zip(FEATURE_COLUMNS, model.feature_importances_)},
    "baseline": HEALTHY_BASELINE,
    "feature_meta": FEATURE_META,
    "metrics": metrics,
    "rate_metrics": rate_metrics,
}

with open(MODEL_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"Saved bundle to {MODEL_PATH.resolve()}")

## 9. Sanity check — reload and predict

In [ ]:
with open(MODEL_PATH, "rb") as f:
    loaded = pickle.load(f)

sample = X_test.iloc[[0]]
p = loaded["model"].predict_proba(sample)[0, 1]
print("Reloaded OK. Sample predicted hot-flash probability:", round(float(p), 3))
print("Bundled metrics:", loaded["metrics"])